In [48]:
import sqlite3

conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = 1")
cursor = conn.cursor()

In [49]:
cursor.executescript('''
CREATE TABLE s(
    sno TEXT PRIMARY KEY,
    sname TEXT,
    status INTEGER CHECK(status BETWEEN 0 AND 255),
    city TEXT CHECK(city IN ('北京','上海','广州','南京','天津','重庆'))
);

CREATE TABLE p(
    pno TEXT PRIMARY KEY,
    pname TEXT,
    color TEXT CHECK(color IN ('红色','黄色','黑色','蓝色','绿色')),
    weight REAL,
    city TEXT CHECK(city IN ('北京','上海','广州','南京','天津','重庆'))
);

CREATE TABLE j(
    jno TEXT PRIMARY KEY,
    jname TEXT,
    city TEXT CHECK(city IN ('北京','上海','广州','南京','天津','重庆'))
);

CREATE TABLE spj(
    sno TEXT REFERENCES s(sno) ON UPDATE CASCADE,
    pno TEXT REFERENCES p(pno) ON UPDATE CASCADE,
    jno TEXT REFERENCES j(jno) ON UPDATE CASCADE,
    qty INTEGER,
    price INTEGER,
    PRIMARY KEY(sno, pno, jno)
);
''')

# 提交事务
conn.commit()

In [50]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("已创建表:", [table[0] for table in cursor.fetchall()])

已创建表: ['s', 'p', 'j', 'spj']


In [51]:
s_data = [
    ('S1', 'Smith', 20, '北京'),
    ('S2', 'Jones', 10, '上海'),
    ('S3', 'Blake', 30, '上海'),
    ('S4', 'Clark', 20, '北京'),
    ('S5', 'Adams', 30, '重庆')
]

p_data = [
    ('P1', 'Nut', '红色', 12.0, '北京'),
    ('P2', 'Bolt', '绿色', 17.0, '上海'),
    ('P3', 'Screw', '蓝色', 17.0, '南京'),
    ('P4', 'Screw', '红色', 14.0, '北京'),
    ('P5', 'Cam', '蓝色', 12.0, '上海'),
    ('P6', 'Cog', '红色', 19.0, '北京')
]

j_data = [
    ('J1', 'Sorter', '上海'),
    ('J2', 'Punch', '南京'),
    ('J3', 'Reader', '重庆'),
    ('J4', 'Console', '重庆'),
    ('J5', 'Collator', '北京'),
    ('J6', 'Terminal', '天津'),
    ('J7', 'Tape', '北京')
]

spj_data = [
    ('S1', 'P1', 'J1', 200, 100),
    ('S1', 'P1', 'J4', 700, 100),
    ('S2', 'P3', 'J1', 400, 10),
    ('S2', 'P3', 'J2', 200, 10),
    ('S2', 'P3', 'J3', 200, 12),
    ('S2', 'P3', 'J4', 500, 10),
    ('S2', 'P3', 'J5', 600, 20),
    ('S2', 'P3', 'J6', 400, 10),
    ('S2', 'P3', 'J7', 800, 8),
    ('S2', 'P5', 'J2', 100, 10),
    ('S3', 'P3', 'J1', 200, 20),
    ('S3', 'P4', 'J2', 500, 18),
    ('S4', 'P6', 'J3', 300, 30),
    ('S4', 'P6', 'J7', 300, 38),
    ('S5', 'P2', 'J2', 200, 40),
    ('S5', 'P2', 'J4', 100, 45),
    ('S5', 'P5', 'J5', 500, 30),
    ('S5', 'P5', 'J7', 100, 30),
    ('S5', 'P6', 'J2', 200, 30),
    ('S5', 'P1', 'J4', 100, 30),
    ('S5', 'P3', 'J4', 200, 30),
    ('S5', 'P4', 'J4', 800, 28),
    ('S5', 'P5', 'J4', 400, 40),
    ('S5', 'P6', 'J4', 500, 29)
]

In [52]:
def insert_data(table, data):
    try:
        cursor.executemany(f'INSERT INTO {table} VALUES ({",".join("?"*len(data[0]))})', data)
        conn.commit()
        print(f"{table} 表插入成功，影响行数：{cursor.rowcount}")
    except sqlite3.IntegrityError as e:
        print(f"违反约束：{str(e)}")

In [53]:
insert_data('s', s_data)
insert_data('p', p_data)
insert_data('j', j_data)
insert_data('spj', spj_data)


s 表插入成功，影响行数：5
p 表插入成功，影响行数：6
j 表插入成功，影响行数：7
spj 表插入成功，影响行数：24


In [54]:
def print_query_result(description, query):
    print(f"\n=== {description} ===")
    cursor.execute(query)
    

    col_names = [desc[0] for desc in cursor.description]
    print(" | ".join(col_names))
    
    for row in cursor.fetchall():
        print(" | ".join(map(str, row)))

## 请完成下列查询，直接在问题的markdown单元格下方插入python单元格写入代码并执行输出答案即可：

### 1.	求向北京的工程供应了红色零件的供应商姓名

In [55]:
query = '''
SELECT DISTINCT sname AS 供应商姓名
FROM s
JOIN spj ON s.sno = spj.sno
JOIN p ON spj.pno = p.pno
JOIN j ON spj.jno = j.jno
WHERE j.city = '北京' AND p.color = '红色'
'''

print_query_result("向北京的工程供应了红色零件的供应商姓名",query)


=== 向北京的工程供应了红色零件的供应商姓名 ===
供应商姓名
Clark


### 2.	求同时供应零件号为P1和P2两种零件的供应商姓名

In [56]:
query = '''
SELECT DISTINCT s.sname AS 供应商姓名
FROM s, spj spj1, spj spj2
WHERE s.sno = spj1.sno
  AND s.sno = spj2.sno
  AND spj1.pno = 'P1'
  AND spj2.pno = 'P2';
'''
print_query_result("同时供应了零件P1和P2的供应商姓名", query)


=== 同时供应了零件P1和P2的供应商姓名 ===
供应商姓名
Adams


### 3.	求没有供应零件号为P1或P2两种零件的供应商姓名

In [57]:
query = '''
SELECT sname AS 供应商姓名
FROM s
WHERE sno NOT IN (
    SELECT sno
    FROM spj
    WHERE pno = 'P1' OR pno = 'P2'
);
'''
print_query_result("未供应零件P1或P2的供应商姓名", query)


=== 未供应零件P1或P2的供应商姓名 ===
供应商姓名
Jones
Blake
Clark


### 4.	列出所有供应商的信息（使用外连接，也包括没有供应零件的供应商）

In [58]:
query = '''
SELECT s.*, spj.pno, spj.jno, spj.qty
FROM s
LEFT JOIN spj ON s.sno = spj.sno;
'''
print_query_result("所有供应商信息（如果有）", query)


=== 所有供应商信息（如果有） ===
sno | sname | status | city | pno | jno | qty
S1 | Smith | 20 | 北京 | P1 | J1 | 200
S1 | Smith | 20 | 北京 | P1 | J4 | 700
S2 | Jones | 10 | 上海 | P3 | J1 | 400
S2 | Jones | 10 | 上海 | P3 | J2 | 200
S2 | Jones | 10 | 上海 | P3 | J3 | 200
S2 | Jones | 10 | 上海 | P3 | J4 | 500
S2 | Jones | 10 | 上海 | P3 | J5 | 600
S2 | Jones | 10 | 上海 | P3 | J6 | 400
S2 | Jones | 10 | 上海 | P3 | J7 | 800
S2 | Jones | 10 | 上海 | P5 | J2 | 100
S3 | Blake | 30 | 上海 | P3 | J1 | 200
S3 | Blake | 30 | 上海 | P4 | J2 | 500
S4 | Clark | 20 | 北京 | P6 | J3 | 300
S4 | Clark | 20 | 北京 | P6 | J7 | 300
S5 | Adams | 30 | 重庆 | P1 | J4 | 100
S5 | Adams | 30 | 重庆 | P2 | J2 | 200
S5 | Adams | 30 | 重庆 | P2 | J4 | 100
S5 | Adams | 30 | 重庆 | P3 | J4 | 200
S5 | Adams | 30 | 重庆 | P4 | J4 | 800
S5 | Adams | 30 | 重庆 | P5 | J4 | 400
S5 | Adams | 30 | 重庆 | P5 | J5 | 500
S5 | Adams | 30 | 重庆 | P5 | J7 | 100
S5 | Adams | 30 | 重庆 | P6 | J2 | 200
S5 | Adams | 30 | 重庆 | P6 | J4 | 500


### 5.	列出向与⾃⼰位于相同城市的⼯程供应零件的供应商姓名

In [59]:
query = '''
SELECT DISTINCT s.sname AS 供应商姓名
FROM s, spj, j
WHERE s.sno = spj.sno
  AND spj.jno = j.jno
  AND s.city = j.city
'''
print_query_result("供应商所在城市与工程所在城市相同的供应商姓名", query)



=== 供应商所在城市与工程所在城市相同的供应商姓名 ===
供应商姓名
Jones
Blake
Clark
Adams


### 6.	列出只向与⾃⼰位于相同城市的⼯程供应零件的供应商姓名

In [60]:
query = '''
SELECT DISTINCT s.sname AS 供应商姓名
FROM s, spj
WHERE s.sno = spj.sno
  AND NOT EXISTS (
      SELECT *
      FROM spj spj2, j
      WHERE spj2.jno = j.jno
        AND spj2.sno = s.sno
        AND s.city <> j.city
  )
'''
print_query_result('只向位于自己所在城市的工程供应零件的供应商姓名', query)


=== 只向位于自己所在城市的工程供应零件的供应商姓名 ===
供应商姓名


### 7.	求供应了所有零件的供应商姓名（使用双重否定）

In [61]:
query = '''
SELECT s.sname AS 供应商姓名
FROM s
WHERE NOT EXISTS (
    SELECT *
    FROM p
    WHERE NOT EXISTS (
        SELECT *
        FROM spj
        WHERE spj.sno = s.sno
          AND spj.pno = p.pno
    )
);
'''
print_query_result("供应了所有零件的供应商姓名", query)


=== 供应了所有零件的供应商姓名 ===
供应商姓名
Adams


### 8.	列出每个城市的⼯程所使⽤的零件总的数量

In [62]:
query = '''
SELECT j.city AS 城市, SUM(spj.qty) AS 零件总数量
FROM j
JOIN spj ON j.jno = spj.jno
GROUP BY j.city
'''
print_query_result("每个城市的工程所需零件总数量", query)


=== 每个城市的工程所需零件总数量 ===
城市 | 零件总数量
上海 | 800
北京 | 2300
南京 | 1200
天津 | 400
重庆 | 3800


### 9.	每项⼯程所使⽤的每种红⾊零件的总的数量

In [63]:
query = '''
SELECT j.jno AS 工程号,
       j.jname AS 工程名,
       COALESCE(SUM(CASE WHEN p.color = '红色' THEN spj.qty ELSE 0 END), 0) AS 红色零件总数量
FROM j
LEFT JOIN spj ON j.jno = spj.jno
LEFT JOIN p ON spj.pno = p.pno
GROUP BY j.jno, j.jname;
'''
print_query_result("每个城市的工程所需红色零件总数量", query)


=== 每个城市的工程所需红色零件总数量 ===
工程号 | 工程名 | 红色零件总数量
J1 | Sorter | 200
J2 | Punch | 700
J3 | Reader | 300
J4 | Console | 2100
J5 | Collator | 0
J6 | Terminal | 0
J7 | Tape | 300


### 10.	供应零件数量最多的供应商姓名

In [64]:
query = '''
SELECT s.sname AS 供应商姓名
FROM s
JOIN spj ON s.sno = spj.sno
GROUP BY s.sno
HAVING SUM(spj.qty) = (
    SELECT MAX(total_qty)
    FROM (
        SELECT SUM(qty) AS total_qty
        FROM spj
        GROUP BY sno
    ) AS t
);
'''
print_query_result("供应零件总数量最多的供应商姓名", query)


=== 供应零件总数量最多的供应商姓名 ===
供应商姓名
Jones


### 11.	每个城市中供应零件数量最多的供应商姓名

In [68]:
query = '''
SELECT j.city AS 城市, s.sname AS 供应商姓名
FROM s
JOIN spj ON s.sno = spj.sno
JOIN j ON spj.jno = j.jno
GROUP BY s.sno, s.sname, j.city
HAVING SUM(spj.qty) = (
    SELECT MAX(total_qty)
    FROM (
        SELECT spj2.sno, j2.city, SUM(spj2.qty) AS total_qty
        FROM spj spj2
        JOIN j j2 ON spj2.jno = j2.jno
        WHERE j2.city = j.city
        GROUP BY spj2.sno, j2.city
    ) AS t
);
'''
print_query_result("每个城市中供应零件总数量最多的供应商姓名", query)


=== 每个城市中供应零件总数量最多的供应商姓名 ===
城市 | 供应商姓名
上海 | Jones
北京 | Jones
天津 | Jones
南京 | Blake
重庆 | Adams
